# Stage 2: Data Preparation (PySpark)
Cleaning, missing values, duplicates, outliers, feature engineering and the train/validation/test split.
The result is written to the **curated zone** of the data lake.

**Prediction task:** predict `fare_amount` (regression). Features are available at booking time (distance estimate, time, zones, passengers, rate code).
Leakage control: tip, tolls, total_amount, duration and payment type are excluded because they are not known before the trip.

In [1]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

BASE = Path('..') if Path('../data').exists() else Path('.')
LAKE = BASE / 'data' / 'lake'
YEAR, MONTHS = 2024, [1, 2, 3]   # must match Stage 1

spark = (SparkSession.builder.appName('nyc-taxi-stage2').master('local[*]')
         .config('spark.driver.memory', '4g').config('spark.sql.shuffle.partitions', '16').getOrCreate())

In [5]:
# ============================================================
# Stage 2: Data Preparation (PySpark)
# Self-contained Colab version
# ============================================================

import os
import requests
from pathlib import Path

BASE = Path('.')
RAW = BASE / 'data' / 'raw'
LAKE = BASE / 'data' / 'lake'
RESULTS = BASE / 'results'

RAW.mkdir(parents=True, exist_ok=True)
LAKE.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

YEAR = 2024
MONTHS = [1, 2, 3]

URL = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{y}-{m:02d}.parquet'
ZONES_URL = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'

print("Setup complete.")
print("BASE:", BASE.resolve())
print("RAW:", RAW.resolve())
print("LAKE:", LAKE.resolve())

Setup complete.
BASE: /content
RAW: /content/data/raw
LAKE: /content/data/lake


In [6]:
# ============================================================
# Stage 2: Data Preparation (PySpark)
# Self-contained Colab version
# ============================================================

import os
import requests
from pathlib import Path

BASE = Path('.')
RAW = BASE / 'data' / 'raw'
LAKE = BASE / 'data' / 'lake'
RESULTS = BASE / 'results'

RAW.mkdir(parents=True, exist_ok=True)
LAKE.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

YEAR = 2024
MONTHS = [1, 2, 3]

URL = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{y}-{m:02d}.parquet'
ZONES_URL = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'

print("Setup complete.")
print("BASE:", BASE.resolve())
print("RAW:", RAW.resolve())
print("LAKE:", LAKE.resolve())

Setup complete.
BASE: /content
RAW: /content/data/raw
LAKE: /content/data/lake


In [7]:
# ============================================================
# 2. Start Spark
# ============================================================

!pip -q install pyspark

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName('nyc-taxi-stage2')
    .master('local[*]')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.shuffle.partitions', '16')
    .getOrCreate()
)

print('Spark:', spark.version)

Spark: 4.0.4


In [10]:
# ============================================================
# 3. DOWNLOAD / VERIFY DATA AND LOAD INTO SPARK
# ============================================================

import os
import requests
from pathlib import Path

# EXACT paths you gave
RAW = Path('/content/data/raw')
LAKE = Path('/content/data/lake')

RAW.mkdir(parents=True, exist_ok=True)
LAKE.mkdir(parents=True, exist_ok=True)

YEAR = 2024
MONTHS = [1, 2, 3]

URL = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{y}-{m:02d}.parquet'

# ------------------------------------------------------------
# Download files if they are not actually present
# ------------------------------------------------------------

def download_file(url, destination):
    destination = Path(destination)

    if destination.exists() and destination.stat().st_size > 0:
        print("Already exists:", destination)
        return

    print("Downloading:", destination.name)

    r = requests.get(url, stream=True, timeout=180)
    r.raise_for_status()

    with open(destination, 'wb') as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

    print(
        "Downloaded:",
        destination,
        round(destination.stat().st_size / 1e6, 1),
        "MB"
    )


for m in MONTHS:
    file_path = RAW / f'yellow_tripdata_{YEAR}-{m:02d}.parquet'

    download_file(
        URL.format(y=YEAR, m=m),
        file_path
    )

# ------------------------------------------------------------
# VERIFY files BEFORE Spark reads them
# ------------------------------------------------------------

print("\nChecking files:")

raw_files = []

for m in MONTHS:
    file_path = RAW / f'yellow_tripdata_{YEAR}-{m:02d}.parquet'

    print(
        file_path,
        "| exists =", file_path.exists(),
        "| size =", round(file_path.stat().st_size / 1e6, 1) if file_path.exists() else 0,
        "MB"
    )

    raw_files.append(str(file_path))

# ------------------------------------------------------------
# READ DATA WITH SPARK
# ------------------------------------------------------------

df_raw = spark.read.parquet(*raw_files)

print("\nRaw rows:", df_raw.count())
print("Columns:", len(df_raw.columns))

# ------------------------------------------------------------
# CREATE RAW DATA LAKE
# ------------------------------------------------------------

df_part = df_raw.withColumn(
    'pickup_month',
    F.date_format('tpep_pickup_datetime', 'yyyy-MM')
)

raw_zone = '/content/data/lake/raw_zone/yellow_trips'

(
    df_part.write
    .mode('overwrite')
    .partitionBy('pickup_month')
    .parquet(raw_zone)
)

print("\nRaw zone created:", raw_zone)

Downloading: yellow_tripdata_2024-01.parquet
Downloaded: /content/data/raw/yellow_tripdata_2024-01.parquet 50.0 MB
Downloading: yellow_tripdata_2024-02.parquet
Downloaded: /content/data/raw/yellow_tripdata_2024-02.parquet 50.3 MB
Downloading: yellow_tripdata_2024-03.parquet
Downloaded: /content/data/raw/yellow_tripdata_2024-03.parquet 60.1 MB

Checking files:
/content/data/raw/yellow_tripdata_2024-01.parquet | exists = True | size = 50.0 MB
/content/data/raw/yellow_tripdata_2024-02.parquet | exists = True | size = 50.3 MB
/content/data/raw/yellow_tripdata_2024-03.parquet | exists = True | size = 60.1 MB

Raw rows: 9554778
Columns: 19

Raw zone created: /content/data/lake/raw_zone/yellow_trips


## 1. Load raw zone and keep relevant columns

In [11]:
raw = spark.read.parquet(str(LAKE / 'raw_zone' / 'yellow_trips'))
cols = ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance',
        'RatecodeID', 'PULocationID', 'DOLocationID', 'fare_amount', 'pickup_month']
df = raw.select(*cols)
report = {'raw_rows': df.count()}
print(report)

{'raw_rows': 9554778}


## 2. Missing values
Rows with nulls in any modelling column are dropped. Imputing fare, distance or location would invent information about the target-related fields, and the null share is small (check Stage 1 output).

In [12]:
df = df.dropna()
report['after_dropna'] = df.count()
print(report)

{'raw_rows': 9554778, 'after_dropna': 8802816}


## 3. Duplicates

In [13]:
df = df.dropDuplicates()
report['after_duplicates'] = df.count()
print(report)

{'raw_rows': 9554778, 'after_dropna': 8802816, 'after_duplicates': 8802483}


## 4. Invalid values and outliers (rule-based)
Rules (each is a documented decision, justify them in the report):
- pickup month must belong to the downloaded months (the raw files contain a few mis-dated records)
- `fare_amount` 3 to 200 USD (3 is roughly the minimum metered fare; negatives are refunds/errors)
- `trip_distance` 0.1 to 100 miles
- `passenger_count` 1 to 6
- `RatecodeID` 1 to 6 (99 is an unknown code)
- zones 1 to 263 (264/265 are the 'unknown' zones)
- duration 1 to 180 minutes and average speed under 80 mph (removes impossible trips)

In [14]:
valid_months = [f'{YEAR}-{m:02d}' for m in MONTHS]
df = (df.withColumn('duration_min', (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / 60)
        .withColumn('speed_mph', F.col('trip_distance') / (F.col('duration_min') / 60)))

df = df.filter(
    F.col('pickup_month').isin(valid_months) &
    F.col('fare_amount').between(3, 200) &
    F.col('trip_distance').between(0.1, 100) &
    F.col('passenger_count').between(1, 6) &
    F.col('RatecodeID').between(1, 6) &
    F.col('PULocationID').between(1, 263) & F.col('DOLocationID').between(1, 263) &
    F.col('duration_min').between(1, 180) &
    (F.col('speed_mph') < 80))
report['after_validity_filters'] = df.count()
print(report)

{'raw_rows': 9554778, 'after_dropna': 8802816, 'after_duplicates': 8802483, 'after_validity_filters': 8273082}


## 5. Feature engineering and encoding
- `hour`, `dow` (0=Mon), `is_weekend` from the pickup timestamp
- cyclical encoding `hour_sin`, `hour_cos` so that 23:00 and 00:00 are close
- zone IDs and rate code stay as integer IDs; the neural network learns **embeddings** for them in Stage 5 (instead of 260+ one-hot columns)
- scaling of numeric features is fitted on the training split only (Stage 4) to avoid leakage

In [15]:
import math
df = (df.withColumn('hour', F.hour('tpep_pickup_datetime'))
        .withColumn('dow', (F.dayofweek('tpep_pickup_datetime') + 5) % 7)   # Spark: Sunday=1 -> convert to Monday=0
        .withColumn('is_weekend', (F.col('dow') >= 5).cast('int'))
        .withColumn('hour_sin', F.sin(2 * math.pi * F.col('hour') / 24))
        .withColumn('hour_cos', F.cos(2 * math.pi * F.col('hour') / 24))
        .withColumn('passenger_count', F.col('passenger_count').cast('int'))
        .withColumn('RatecodeID', F.col('RatecodeID').cast('int'))
        .withColumn('PULocationID', F.col('PULocationID').cast('int'))
        .withColumn('DOLocationID', F.col('DOLocationID').cast('int'))
        .withColumn('trip_distance', F.col('trip_distance').cast('double'))
        .withColumn('fare_amount', F.col('fare_amount').cast('double')))

final_cols = ['trip_distance', 'passenger_count', 'hour', 'dow', 'is_weekend', 'hour_sin', 'hour_cos',
              'PULocationID', 'DOLocationID', 'RatecodeID', 'fare_amount', 'pickup_month']
df = df.select(*final_cols)
df.show(5)

+-------------+---------------+----+---+----------+--------+--------+------------+------------+----------+-----------+------------+
|trip_distance|passenger_count|hour|dow|is_weekend|hour_sin|hour_cos|PULocationID|DOLocationID|RatecodeID|fare_amount|pickup_month|
+-------------+---------------+----+---+----------+--------+--------+------------+------------+----------+-----------+------------+
|          2.4|              1|   0|  4|         0|     0.0|     1.0|         237|         234|         1|       12.8|     2024-03|
|         1.24|              1|   0|  4|         0|     0.0|     1.0|         161|         170|         1|        9.3|     2024-03|
|         1.92|              1|   0|  4|         0|     0.0|     1.0|         162|          90|         1|       12.8|     2024-03|
|         0.67|              1|   0|  4|         0|     0.0|     1.0|         231|         211|         1|        5.8|     2024-03|
|          3.0|              1|   0|  4|         0|     0.0|     1.0|       

## 6. Train / validation / test split (70 / 15 / 15)
Random split with a fixed seed. Test data is touched only once, in Stage 6.

In [16]:
df = df.withColumn('_r', F.rand(42))
train = df.filter(F.col('_r') < 0.70).drop('_r')
val   = df.filter((F.col('_r') >= 0.70) & (F.col('_r') < 0.85)).drop('_r')
test  = df.filter(F.col('_r') >= 0.85).drop('_r')

cz = str(LAKE / 'curated_zone')
train.write.mode('overwrite').parquet(cz + '/train')
val.write.mode('overwrite').parquet(cz + '/val')
test.write.mode('overwrite').parquet(cz + '/test')

for name in ['train', 'val', 'test']:
    report[name + '_rows'] = spark.read.parquet(f'{cz}/{name}').count()
print(report)

{'raw_rows': 9554778, 'after_dropna': 8802816, 'after_duplicates': 8802483, 'after_validity_filters': 8273082, 'train_rows': 5791453, 'val_rows': 1239884, 'test_rows': 1241745}


In [17]:
import json, pandas as pd
(BASE / 'results').mkdir(exist_ok=True)
(BASE / 'results' / 'cleaning_report.json').write_text(json.dumps(report, indent=2))
pd.Series(report, name='rows').to_frame()

,rows
raw_rows,9554778
after_dropna,8802816
after_duplicates,8802483
after_validity_filters,8273082
train_rows,5791453
val_rows,1239884
test_rows,1241745


## Notes for the report
- Copy the cleaning report table (rows removed at each step and percentage).
- Explain every threshold above in one sentence.
- Next: Stage 3, Big Data processing and analysis with Spark.